# Phase 3 : Analyse Exploratoire des Données (EDA)

**Objectif :** Comprendre les données à travers des visualisations et des statistiques.

## Questions :
- Combien de films et d'utilisateurs ?
- Quelle est la distribution des notes ?
- Quels sont les genres les plus populaires ?
- Quels sont les films les mieux notés ?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration des graphiques
plt.style.use('ggplot')
sns.set_palette('Set2')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

# Charger les données
data = pd.read_csv('../data/ratings_with_movies.csv')
print('✅ Données chargées')

In [ ]:
# STATISTIQUES GÉNÉRALES
n_films = data['movieId'].nunique()
n_users = data['userId'].nunique()
n_ratings = len(data)
sparsity = n_ratings / (n_films * n_users) * 100

print(f'📊 Nombre de films : {n_films}')
print(f'👥 Nombre d\'utilisateurs : {n_users}')
print(f'⭐ Nombre d\'évaluations : {n_ratings}')
print(f'📉 Densité de la matrice : {sparsity:.3f}%')
print(f'\n📈 Statistiques des notes :')
print(data['rating'].describe())

In [ ]:
# DISTRIBUTION DES NOTES
plt.figure(figsize=(10, 6))
ax = sns.countplot(x='rating', data=data, palette='viridis')
plt.title('Distribution des notes', fontsize=16, fontweight='bold')
plt.xlabel('Note', fontsize=12)
plt.ylabel("Nombre d'évaluations", fontsize=12)

# Ajouter les valeurs sur les barres
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# TOP 10 DES FILMS LES MIEUX NOTÉS
movie_stats = data.groupby('title').agg(
    note_moyenne=('rating', 'mean'),
    nb_evaluations=('rating', 'count')
).reset_index()

# Filtrer les films avec >= 50 évaluations
movie_stats = movie_stats[movie_stats['nb_evaluations'] >= 50]
top_rated = movie_stats.sort_values('note_moyenne', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='note_moyenne', y='title', data=top_rated, palette='rocket')
plt.title('Top 10 des films les mieux notés (≥ 50 évaluations)', fontsize=16, fontweight='bold')
plt.xlabel('Note moyenne', fontsize=12)
plt.ylabel('')
plt.tight_layout()
plt.show()

display(top_rated)

In [ ]:
# DISTRIBUTION DES GENRES
genres_list = data['genres'].str.split('|')
all_genres = [genre for sublist in genres_list for genre in sublist]
genre_counts = pd.Series(all_genres).value_counts()

plt.figure(figsize=(12, 6))
sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='mako')
plt.title('Distribution des genres', fontsize=16, fontweight='bold')
plt.xlabel('Nombre de films', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.tight_layout()
plt.show()

print('\nTop 10 des genres :')
print(genre_counts.head(10))

In [ ]:
# ACTIVITÉ DES UTILISATEURS
user_activity = data.groupby('userId').size()

plt.figure(figsize=(12, 6))
plt.hist(user_activity, bins=50, color='steelblue', edgecolor='white')
plt.axvline(user_activity.mean(), color='red', linestyle='--',
            label=f'Moyenne : {user_activity.mean():.1f}')
plt.axvline(user_activity.median(), color='green', linestyle='--',
            label=f'Médiane : {user_activity.median():.1f}')
plt.title('Distribution du nombre d\'évaluations par utilisateur', fontsize=16, fontweight='bold')
plt.xlabel('Nombre d\'évaluations', fontsize=12)
plt.ylabel('Nombre d\'utilisateurs', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

print(f'Min : {user_activity.min()}, Max : {user_activity.max()}')

In [ ]:
# FILMS LES PLUS NOTÉS
most_rated = movie_stats.sort_values('nb_evaluations', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='nb_evaluations', y='title', data=most_rated, palette='crest')
plt.title('Top 10 des films les plus évalués', fontsize=16, fontweight='bold')
plt.xlabel("Nombre d'évaluations", fontsize=12)
plt.ylabel('')
plt.tight_layout()
plt.show()

display(most_rated)

In [ ]:
# NOTE MOYENNE PAR GENRE
genre_dummies = data['genres'].str.get_dummies('|')
genre_ratings = pd.concat([genre_dummies, data['rating']], axis=1)
genre_avg = genre_ratings.drop('(no genres listed)', axis=1, errors='ignore').mean()

plt.figure(figsize=(10, 6))
genre_avg.sort_values().plot(kind='barh', color='coral')
plt.title('Note moyenne par genre', fontsize=16, fontweight='bold')
plt.xlabel('Note moyenne', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# SYNTHÈSE
print('='*60)
print("SYNTHÈSE DE L'ANALYSE EXPLORATOIRE")
print('='*60)
print(f'\n📊 Le dataset contient {n_films} films notés par {n_users} utilisateurs.')
print(f'⭐ La note moyenne globale est de {data["rating"].mean():.2f}/5.')
print(f'🎬 Le film le plus évalué est : {most_rated.iloc[0]["title"]}')
print(f'🎭 Le genre le plus représenté est : {genre_counts.index[0]}')
print(f'📈 La densité de la matrice est de {sparsity:.3f}%.')
print(f'👉 Cette faible densité justifie l\'utilisation du filtrage collaboratif.')